In [ ]:
    # -*- coding: utf-8 -*-
    """
    TESTE — VARIAÇÕES DO RF LIVRE vs PARK EM JANELAS PEQUENAS PERTO DOS PICOS
    ================================================================================
    
    Ideia deste código
    ------------------
    Você testou o RF livre antigo contra Park e viu que Park ganhou no multiclasse
    D0/D1/D2, mas o RF livre pareceu interessante para detectar dano, porque quase
    não jogou dano para a classe saudável.
    
    Este script repete o teste, mas agora avalia várias configurações do RF livre:
    
        - RF_original: mesmos parâmetros do seu RF antigo;
        - RF_raso: menos profundidade, correção menos agressiva;
        - RF_regularizado: folhas maiores e split maior;
        - RF_profundo: mais liberdade, para ver se amplifica mais dano;
        - RF_muitas_arvores: igual ao original, mas com mais árvores;
        - Park: comparação física/conservadora.
    
    O objetivo é responder:
    
        1) algum RF parametrizado supera o Park no multiclasse D0/D1/D2?
        2) algum RF é melhor para o teste binário sem dano vs com dano?
        3) o RF reduz falsos saudáveis? isto é: dano real previsto como D0.
    
    Validação
    ---------
    O teste usa Leave-One-Temperature-Out:
    
        - escolhe uma temperatura como teste;
        - treina o classificador nas outras temperaturas;
        - testa na temperatura deixada fora.
    
    Por padrão, o compensador RF também é treinado sem usar amostras saudáveis da
    temperatura de teste, para evitar vazamento otimista.
    
    Saídas principais
    -----------------
    Pasta: resultados_RF_variantes_vs_Park_picos
    
    CSV:
        - janelas_picos_selecionadas.csv
        - resultados_folds_multiclasse.csv
        - resultados_folds_binario.csv
        - resumo_multiclasse_por_janela.csv
        - resumo_binario_por_janela.csv
        - metricas_teste_por_amostra.csv
        - predicoes_multiclasse.csv
        - predicoes_binario.csv
    
    Gráficos:
        - ranking multiclasse Macro-F1;
        - ranking binário recall de dano e falso saudável;
        - matriz de confusão multiclasse;
        - matriz de confusão binária;
        - RMSD/CCDM por temperatura;
        - curvas exemplo Original × RF variants × Park × referência.
    """
    
    # ============================================================
    # 1) IMPORTS
    # ============================================================
    
    import os
    import re
    import time
    import warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        f1_score,
        recall_score,
        precision_score,
        confusion_matrix,
    )
    
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    np.random.seed(42)
    
    
    # ============================================================
    # 2) CONFIGURAÇÕES GERAIS
    # ============================================================
    
    ARQ_BASE = "base-completo--.pkl"
    
    PASTA_SAIDA = "resultados_RF_variantes_vs_Park_picos"
    PASTA_CSV = os.path.join(PASTA_SAIDA, "csvs")
    PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos")
    PASTA_CONFUSAO = os.path.join(PASTA_GRAFICOS, "matrizes_confusao")
    PASTA_CURVAS = os.path.join(PASTA_GRAFICOS, "curvas_exemplo")
    PASTA_METRICAS = os.path.join(PASTA_GRAFICOS, "metricas_por_temperatura")
    
    for p in [PASTA_SAIDA, PASTA_CSV, PASTA_GRAFICOS, PASTA_CONFUSAO, PASTA_CURVAS, PASTA_METRICAS]:
        os.makedirs(p, exist_ok=True)
    
    # Temperatura de referência da curva saudável.
    REF_TEMP = 30
    
    # Busca de picos/vales.
    FREQ_PEAK_SEARCH_MIN_KHZ = 30
    FREQ_PEAK_SEARCH_MAX_KHZ = 100
    
    # Janelas pequenas próximas dos picos.
    PEAK_WINDOW_WIDTH_KHZ = 3.0
    TOP_N_PEAK_WINDOWS = 10
    MIN_DIST_ENTRE_PICOS_KHZ = 3.0
    MIN_FREQ_POINTS_WINDOW = 8
    
    # Para forçar janelas manualmente, coloque centros aqui.
    # Exemplo: [40.2] gera janela em torno de 40.2 kHz.
    PEAK_CENTERS_MANUAL_KHZ = []
    
    # Também pode forçar uma janela fixa J2 parecida com seu resultado.
    # Se True, adiciona 38.7–41.7 kHz mesmo que a detecção automática não escolha.
    INCLUIR_JANELA_J2_MANUAL = True
    JANELA_J2_MANUAL = (38.7, 41.7)
    
    # Suavização aplicada depois da compensação do RF.
    # O RF original usava 5. Aqui cada variante pode sobrescrever isso.
    SMOOTH_WIN_PADRAO = 5
    
    # Modo de validação.
    STRICT_LOTO_COMPENSATION = True
    
    # Para comparar exatamente com versões anteriores, deixe True.
    # Se colocar False, a referência saudável também é montada dentro de cada fold,
    # usando só as temperaturas de treino. É mais rigoroso, mas muda os resultados.
    USAR_REFERENCIA_GLOBAL_DA_JANELA = True
    
    # Classificação.
    FEATURE_SETS = ["metricas_RMSD_CCDM", "curva_e_metricas"]
    FEATURE_SET_PREFERIDO = "metricas_RMSD_CCDM"
    
    # Temperatura para curvas exemplo.
    TEMP_EXEMPLO_CURVAS = 80
    
    SALVAR_PDF = True
    
    DANOS = [0, 1, 2]
    LABELS_MULTI = [0, 1, 2]
    LABELS_MULTI_TXT = ["D0", "D1", "D2"]
    LABELS_BIN = [0, 1]
    LABELS_BIN_TXT = ["Sem dano", "Com dano"]
    
    
    # ============================================================
    # 3) VARIANTES DO RF LIVRE
    # ============================================================
    # A ideia é testar se regularizar o RF melhora a generalização:
    # - menor profundidade e folhas maiores deixam a correção menos agressiva;
    # - maior profundidade testa a hipótese de amplificação discriminativa;
    # - mais árvores testa estabilidade sem mudar a lógica.
    
    RF_VARIANTS = {
        "RF_original": {
            "nome": "RF original",
            "smooth_win": 5,
            "params": dict(
                n_estimators=250,
                max_depth=10,
                min_samples_leaf=2,
                min_samples_split=4,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    
        "RF_raso": {
            "nome": "RF raso",
            "smooth_win": 5,
            "params": dict(
                n_estimators=250,
                max_depth=6,
                min_samples_leaf=4,
                min_samples_split=8,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    
        "RF_regularizado": {
            "nome": "RF regularizado",
            "smooth_win": 5,
            "params": dict(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=6,
                min_samples_split=12,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    
        "RF_profundo": {
            "nome": "RF profundo",
            "smooth_win": 5,
            "params": dict(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=1,
                min_samples_split=2,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    
        "RF_muitas_arvores": {
            "nome": "RF muitas árvores",
            "smooth_win": 5,
            "params": dict(
                n_estimators=600,
                max_depth=10,
                min_samples_leaf=2,
                min_samples_split=4,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    
        "RF_regularizado_suave": {
            "nome": "RF regularizado + suave",
            "smooth_win": 9,
            "params": dict(
                n_estimators=300,
                max_depth=7,
                min_samples_leaf=8,
                min_samples_split=16,
                max_features="sqrt",
                n_jobs=-1,
                random_state=0,
            )
        },
    }
    
    # Park.
    PARK_MAX_SHIFT_FRAC = 0.25
    PARK_SMOOTH_WIN = 5
    PARK_NSTEPS = 151
    
    METODOS_RF = list(RF_VARIANTS.keys())
    METODOS_TODOS = METODOS_RF + ["Park"]
    
    NOME_METODO = {k: v["nome"] for k, v in RF_VARIANTS.items()}
    NOME_METODO["Park"] = "Park"
    
    
    # ============================================================
    # 4) ESTILO VISUAL
    # ============================================================
    
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 19,
        "axes.labelsize": 23,
        "axes.titlesize": 24,
        "xtick.labelsize": 16,
        "ytick.labelsize": 17,
        "legend.fontsize": 13,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })
    
    CORES_METODO = {
        "RF_original": "tab:blue",
        "RF_raso": "tab:cyan",
        "RF_regularizado": "tab:purple",
        "RF_profundo": "tab:red",
        "RF_muitas_arvores": "tab:orange",
        "RF_regularizado_suave": "tab:brown",
        "Park": "tab:green",
    }
    
    CORES_DANO = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red",
    }
    
    
    # ============================================================
    # 5) FUNÇÕES BÁSICAS
    # ============================================================
    
    def salvar_fig(fig, nome_base, pasta=PASTA_GRAFICOS):
        os.makedirs(pasta, exist_ok=True)
        png = os.path.join(pasta, nome_base + ".png")
        fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    
        if SALVAR_PDF:
            pdf = os.path.join(pasta, nome_base + ".pdf")
            fig.savefig(pdf, bbox_inches="tight", facecolor="white")
            print(f"✅ Salvo:\n{png}\n{pdf}")
        else:
            print(f"✅ Salvo:\n{png}")
    
    
    def estilo_eixos(ax):
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both")
    
    
    def format_temp(T):
        T = float(T)
        if T.is_integer():
            return str(int(T))
        return f"{T:.1f}"
    
    
    def format_faixa(fmin, fmax):
        return f"{fmin:.1f}–{fmax:.1f} kHz"
    
    
    def safe_name(s):
        return str(s).replace(" ", "_").replace("–", "-").replace("/", "_").replace("+", "p")
    
    
    def extract_freq_hz(col):
        m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
        return float(m.group(1)) if m else None
    
    
    def get_freq_columns(df, fmin_khz, fmax_khz):
        cols, freqs = [], []
        for c in df.columns:
            f = extract_freq_hz(c)
            if f is not None:
                f_khz = f / 1e3
                if fmin_khz <= f_khz <= fmax_khz:
                    cols.append(c)
                    freqs.append(f)
        if len(cols) == 0:
            return [], np.array([], dtype=float)
        order = np.argsort(freqs)
        cols = [cols[i] for i in order]
        freqs = np.array(freqs, dtype=float)[order]
        return cols, freqs
    
    
    def moving_average(arr, win):
        arr = np.asarray(arr, dtype=float)
        if win <= 1 or win % 2 == 0:
            return arr.copy()
        pad = win // 2
        arr_pad = np.pad(arr, (pad, pad), mode="edge")
        kernel = np.ones(win, dtype=float) / win
        smooth = np.convolve(arr_pad, kernel, mode="valid")
        if len(smooth) > len(arr):
            smooth = smooth[:len(arr)]
        elif len(smooth) < len(arr):
            smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")
        return smooth
    
    
    def add_extra_features_matrix(X):
        X = np.asarray(X, dtype=float)
        mu = X.mean(axis=1, keepdims=True)
        sd = X.std(axis=1, keepdims=True)
        amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
        return np.hstack([X, mu, sd, amp])
    
    
    def rmsd(y, ref):
        y = np.asarray(y, dtype=float)
        ref = np.asarray(ref, dtype=float)
        return float(np.sqrt(np.mean((y - ref) ** 2)))
    
    
    def ccdm(y, ref):
        y = np.asarray(y, dtype=float)
        ref = np.asarray(ref, dtype=float)
        y0 = y - np.mean(y)
        r0 = ref - np.mean(ref)
        num = float(np.sum(y0 * r0))
        den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18
        corr = num / den
        return float(1 - corr)
    
    
    def curva_referencia_saudavel(df, fcols, ref_temp=REF_TEMP):
        df_sem = df[df["falha"] == 0].copy()
        if len(df_sem) == 0:
            raise ValueError("Não há amostras sem dano para montar referência.")
    
        temps_sem = np.asarray(sorted(df_sem["temperatura_c"].dropna().unique()), dtype=float)
        if len(temps_sem) == 0:
            raise ValueError("Não há temperaturas válidas no conjunto sem dano.")
    
        if np.any(np.isclose(temps_sem, ref_temp)):
            temp_usada = float(temps_sem[np.where(np.isclose(temps_sem, ref_temp))[0][0]])
        else:
            temp_usada = float(temps_sem[np.argmin(np.abs(temps_sem - ref_temp))])
            print(f"⚠️ Não achei sem dano em {ref_temp}°C. Usando referência saudável em {temp_usada}°C.")
    
        pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], temp_usada), fcols].to_numpy(float)
        return np.median(pool, axis=0), temp_usada
    
    
    # ============================================================
    # 6) COMPENSAÇÃO RF E PARK
    # ============================================================
    
    def compensar_rf_variante(df, fcols, y_ref, metodo_rf, mask_treino_sem_dano=None):
        cfg = RF_VARIANTS[metodo_rf]
        params = cfg["params"]
        smooth_win = int(cfg.get("smooth_win", SMOOTH_WIN_PADRAO))
    
        if mask_treino_sem_dano is None:
            mask_treino_sem_dano = (df["falha"].to_numpy(int) == 0)
        else:
            mask_treino_sem_dano = np.asarray(mask_treino_sem_dano, dtype=bool)
    
        df_sem = df.loc[mask_treino_sem_dano].copy()
        if len(df_sem) < 3:
            raise ValueError(f"Poucas amostras sem dano para treinar {metodo_rf}.")
    
        X_sem = df_sem[fcols].to_numpy(float)
        T_sem = df_sem["temperatura_c"].to_numpy(float)
    
        Y_target = y_ref[None, :] - X_sem
    
        X_in = np.hstack([
            add_extra_features_matrix(X_sem),
            T_sem.reshape(-1, 1),
        ])
    
        rf = RandomForestRegressor(**params)
        rf.fit(X_in, Y_target)
    
        X_all = df[fcols].to_numpy(float)
        T_all = df["temperatura_c"].to_numpy(float)
        X_all_in = np.hstack([
            add_extra_features_matrix(X_all),
            T_all.reshape(-1, 1),
        ])
    
        delta_pred = rf.predict(X_all_in)
        Y_comp = X_all + delta_pred
    
        for i in range(len(Y_comp)):
            Y_comp[i] = moving_average(Y_comp[i], smooth_win)
    
        df2 = df.copy()
        df2[fcols] = Y_comp
        return df2
    
    
    def shift_interp(x, f, tau):
        f_shift = f + tau
        return np.interp(f, f_shift, x, left=x[0], right=x[-1])
    
    
    def park_single(x, ref, fHz):
        df_band = fHz[-1] - fHz[0]
        tau_max = PARK_MAX_SHIFT_FRAC * df_band
    
        best_err = np.inf
        best_tau = 0.0
        best_dS = 0.0
    
        for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
            x_shift = shift_interp(x, fHz, tau)
            dS = np.mean(ref - x_shift)
            err = np.sum((ref - (x_shift + dS)) ** 2)
            if err < best_err:
                best_err = err
                best_tau = tau
                best_dS = dS
    
        y = shift_interp(x, fHz, best_tau) + best_dS
        y = moving_average(y, PARK_SMOOTH_WIN)
        return y
    
    
    def compensar_park(df, fcols, fHz, y_ref):
        X_all = df[fcols].to_numpy(float)
        Y = np.zeros_like(X_all)
        for i in range(len(X_all)):
            Y[i] = park_single(X_all[i], y_ref, fHz)
        df2 = df.copy()
        df2[fcols] = Y
        return df2
    
    
    def calcular_metricas_df(df_comp, fcols, y_ref):
        X = df_comp[fcols].to_numpy(float)
        out = df_comp[["temperatura_c", "falha"]].copy()
        out["RMSD"] = [rmsd(x, y_ref) for x in X]
        out["CCDM"] = [ccdm(x, y_ref) for x in X]
        return out
    
    
    # ============================================================
    # 7) PICOS, VALES E JANELAS
    # ============================================================
    
    def detectar_picos_vales(df_base):
        fcols_global, fHz_global = get_freq_columns(
            df_base,
            FREQ_PEAK_SEARCH_MIN_KHZ,
            FREQ_PEAK_SEARCH_MAX_KHZ,
        )
    
        if len(fcols_global) < 10:
            raise ValueError("Poucos pontos na região de busca de picos.")
    
        y_ref_global, temp_ref_usada = curva_referencia_saudavel(df_base, fcols_global, REF_TEMP)
        fkhz_global = fHz_global / 1e3
    
        win = 11 if len(y_ref_global) >= 15 else 5
        if win % 2 == 0:
            win += 1
        y_s = moving_average(y_ref_global, win)
        y_norm = (y_s - np.mean(y_s)) / (np.std(y_s) + 1e-18)
    
        candidatos = []
    
        if len(PEAK_CENTERS_MANUAL_KHZ) > 0:
            for c in PEAK_CENTERS_MANUAL_KHZ:
                candidatos.append({"centro_khz": float(c), "tipo": "manual", "score": 999.0})
        else:
            try:
                from scipy.signal import find_peaks
                dfreq = float(np.median(np.diff(fkhz_global))) if len(fkhz_global) > 2 else 0.1
                min_dist_pts = max(1, int(round(MIN_DIST_ENTRE_PICOS_KHZ / max(dfreq, 1e-9))))
    
                peaks, prop_p = find_peaks(y_norm, distance=min_dist_pts, prominence=0.10)
                valleys, prop_v = find_peaks(-y_norm, distance=min_dist_pts, prominence=0.10)
    
                for idx, prom in zip(peaks, prop_p.get("prominences", np.ones(len(peaks)))):
                    candidatos.append({"centro_khz": float(fkhz_global[idx]), "tipo": "pico", "score": float(prom)})
                for idx, prom in zip(valleys, prop_v.get("prominences", np.ones(len(valleys)))):
                    candidatos.append({"centro_khz": float(fkhz_global[idx]), "tipo": "vale", "score": float(prom)})
    
            except Exception:
                dy = np.diff(y_norm)
                sign = np.sign(dy)
                for i in range(1, len(sign)):
                    if sign[i - 1] > 0 and sign[i] < 0:
                        local = y_norm[max(0, i - 10):min(len(y_norm), i + 11)]
                        score = abs(y_norm[i] - np.median(local))
                        candidatos.append({"centro_khz": float(fkhz_global[i]), "tipo": "pico", "score": float(score)})
                    if sign[i - 1] < 0 and sign[i] > 0:
                        local = y_norm[max(0, i - 10):min(len(y_norm), i + 11)]
                        score = abs(y_norm[i] - np.median(local))
                        candidatos.append({"centro_khz": float(fkhz_global[i]), "tipo": "vale", "score": float(score)})
    
        if len(candidatos) == 0:
            raise RuntimeError("Não consegui detectar picos/vales. Use PEAK_CENTERS_MANUAL_KHZ.")
    
        cand = pd.DataFrame(candidatos).sort_values("score", ascending=False).reset_index(drop=True)
    
        selecionados = []
        for _, r in cand.iterrows():
            c = float(r["centro_khz"])
            if c < FREQ_PEAK_SEARCH_MIN_KHZ or c > FREQ_PEAK_SEARCH_MAX_KHZ:
                continue
            if all(abs(c - s["centro_khz"]) >= MIN_DIST_ENTRE_PICOS_KHZ for s in selecionados):
                selecionados.append(r.to_dict())
            if len(selecionados) >= TOP_N_PEAK_WINDOWS:
                break
    
        half = PEAK_WINDOW_WIDTH_KHZ / 2.0
        rows = []
    
        for r in selecionados:
            centro = float(r["centro_khz"])
            fmin = max(FREQ_PEAK_SEARCH_MIN_KHZ, centro - half)
            fmax = min(FREQ_PEAK_SEARCH_MAX_KHZ, centro + half)
            fcols, _ = get_freq_columns(df_base, fmin, fmax)
            if len(fcols) < MIN_FREQ_POINTS_WINDOW:
                continue
            rows.append({
                "centro_khz": centro,
                "faixa_min_khz": float(fmin),
                "faixa_max_khz": float(fmax),
                "largura_khz": float(fmax - fmin),
                "tipo_extremo": r["tipo"],
                "score_extremo": float(r["score"]),
                "n_freq_points": int(len(fcols)),
                "faixa_label": format_faixa(fmin, fmax),
            })
    
        if INCLUIR_JANELA_J2_MANUAL:
            fmin, fmax = JANELA_J2_MANUAL
            fcols, _ = get_freq_columns(df_base, fmin, fmax)
            if len(fcols) >= MIN_FREQ_POINTS_WINDOW:
                rows.append({
                    "centro_khz": 0.5 * (fmin + fmax),
                    "faixa_min_khz": float(fmin),
                    "faixa_max_khz": float(fmax),
                    "largura_khz": float(fmax - fmin),
                    "tipo_extremo": "manual_J2",
                    "score_extremo": 9999.0,
                    "n_freq_points": int(len(fcols)),
                    "faixa_label": format_faixa(fmin, fmax),
                })
    
        df_windows = pd.DataFrame(rows)
        if len(df_windows) == 0:
            raise RuntimeError("Nenhuma janela válida foi gerada.")
    
        # Remove duplicatas quase iguais.
        df_windows["fmin_round"] = df_windows["faixa_min_khz"].round(3)
        df_windows["fmax_round"] = df_windows["faixa_max_khz"].round(3)
        df_windows = df_windows.sort_values(["score_extremo"], ascending=False)
        df_windows = df_windows.drop_duplicates(["fmin_round", "fmax_round"], keep="first")
        df_windows = df_windows.drop(columns=["fmin_round", "fmax_round"])
        df_windows = df_windows.sort_values("centro_khz").reset_index(drop=True)
        df_windows["janela_id"] = np.arange(len(df_windows))
    
        df_windows.to_csv(os.path.join(PASTA_CSV, "janelas_picos_selecionadas.csv"), index=False)
        return df_windows, fkhz_global, y_ref_global, temp_ref_usada
    
    
    def plot_janelas_picos(df_windows, fkhz_global, y_ref_global, temp_ref_usada):
        fig, ax = plt.subplots(figsize=(17, 8), dpi=300)
        ax.plot(fkhz_global, y_ref_global, color="black", linewidth=1.8, label=f"Referência saudável {format_temp(temp_ref_usada)}°C")
    
        for _, r in df_windows.iterrows():
            ax.axvspan(r["faixa_min_khz"], r["faixa_max_khz"], alpha=0.16)
            ax.axvline(r["centro_khz"], color="black", linewidth=0.8, alpha=0.35)
            ax.text(r["centro_khz"], np.nanmax(y_ref_global), f"J{int(r['janela_id'])}\n{r['centro_khz']:.1f}",
                    ha="center", va="top", fontsize=10)
    
        ax.set_xlabel("Frequência (kHz)")
        ax.set_ylabel("Impedância")
        ax.set_title("Janelas pequenas selecionadas perto de picos/vales da referência saudável", pad=14)
        ax.legend(frameon=True, loc="best")
        estilo_eixos(ax)
        fig.tight_layout()
        salvar_fig(fig, "janelas_pequenas_perto_dos_picos")
        plt.show()
    
    
    # ============================================================
    # 8) CLASSIFICAÇÃO
    # ============================================================
    
    def get_valid_temperatures_for_loto(df):
        temps = []
        for T in sorted(df["temperatura_c"].dropna().unique()):
            ok = True
            for d in DANOS:
                if not np.any(np.isclose(df["temperatura_c"], T) & (df["falha"] == d)):
                    ok = False
                    break
            if ok:
                temps.append(float(T))
        return temps
    
    
    def build_features(df_comp, fcols, df_metrics, feature_set):
        if feature_set == "metricas_RMSD_CCDM":
            return df_metrics[["RMSD", "CCDM"]].to_numpy(float)
    
        if feature_set == "curva_e_metricas":
            X_curve = df_comp[fcols].to_numpy(float)
            X_metric = df_metrics[["RMSD", "CCDM"]].to_numpy(float)
            return np.hstack([X_curve, X_metric])
    
        raise ValueError(f"feature_set desconhecido: {feature_set}")
    
    
    def make_classifier():
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                solver="lbfgs",
                random_state=0,
            ))
        ])
    
    
    def avaliar_uma_janela(df_base, row_window):
        fmin = float(row_window["faixa_min_khz"])
        fmax = float(row_window["faixa_max_khz"])
        janela_id = int(row_window["janela_id"])
        faixa_label = str(row_window["faixa_label"])
    
        fcols, fHz = get_freq_columns(df_base, fmin, fmax)
        if len(fcols) < MIN_FREQ_POINTS_WINDOW:
            raise ValueError(f"Poucos pontos na janela {faixa_label}")
    
        df_use = df_base[["temperatura_c", "falha"] + fcols].copy().reset_index(drop=True)
    
        y_ref_global, temp_ref_global = curva_referencia_saudavel(df_use, fcols, REF_TEMP)
    
        temps_loto = get_valid_temperatures_for_loto(df_use)
        if len(temps_loto) < 2:
            raise ValueError(f"Poucas temperaturas válidas para LOTO na janela {faixa_label}")
    
        resultados_multi = []
        resultados_bin = []
        metricas_teste_rows = []
        pred_multi_rows = []
        pred_bin_rows = []
    
        acumulados_multi = {(m, fs): {"y_true": [], "y_pred": []} for m in METODOS_TODOS for fs in FEATURE_SETS}
        acumulados_bin = {(m, fs): {"y_true": [], "y_pred": []} for m in METODOS_TODOS for fs in FEATURE_SETS}
    
        for T_test in temps_loto:
            test_mask = np.isclose(df_use["temperatura_c"].to_numpy(float), T_test)
            train_mask = ~test_mask
    
            y_multi = df_use["falha"].to_numpy(int)
            y_bin = (y_multi > 0).astype(int)
    
            y_train_multi = y_multi[train_mask]
            y_test_multi = y_multi[test_mask]
            y_train_bin = y_bin[train_mask]
            y_test_bin = y_bin[test_mask]
    
            if len(np.unique(y_train_multi)) < 3 or len(np.unique(y_test_multi)) < 2:
                continue
            if len(np.unique(y_train_bin)) < 2 or len(np.unique(y_test_bin)) < 2:
                continue
    
            # Referência: global da janela, para manter comparação com seu teste anterior.
            # Se quiser mais rigor, usa referência só das temperaturas de treino.
            if USAR_REFERENCIA_GLOBAL_DA_JANELA:
                y_ref = y_ref_global
                temp_ref_usada = temp_ref_global
            else:
                y_ref, temp_ref_usada = curva_referencia_saudavel(df_use.loc[train_mask].copy(), fcols, REF_TEMP)
    
            if STRICT_LOTO_COMPENSATION:
                mask_treino_rf = train_mask & (df_use["falha"].to_numpy(int) == 0)
            else:
                mask_treino_rf = (df_use["falha"].to_numpy(int) == 0)
    
            dados_metodos = {}
    
            for metodo_rf in METODOS_RF:
                df_rf = compensar_rf_variante(df_use, fcols, y_ref, metodo_rf, mask_treino_sem_dano=mask_treino_rf)
                met_rf = calcular_metricas_df(df_rf, fcols, y_ref)
                dados_metodos[metodo_rf] = (df_rf, met_rf)
    
            df_pk = compensar_park(df_use, fcols, fHz, y_ref)
            met_pk = calcular_metricas_df(df_pk, fcols, y_ref)
            dados_metodos["Park"] = (df_pk, met_pk)
    
            for metodo, (df_comp, df_met) in dados_metodos.items():
                df_test_met = df_met.loc[test_mask].copy()
                for idx_local, r in df_test_met.iterrows():
                    metricas_teste_rows.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "fold_test_temp": float(T_test),
                        "temperatura_c": float(r["temperatura_c"]),
                        "falha": int(r["falha"]),
                        "classe_binaria": int(int(r["falha"]) > 0),
                        "RMSD": float(r["RMSD"]),
                        "CCDM": float(r["CCDM"]),
                        "temp_ref_usada": float(temp_ref_usada),
                    })
    
                for fs in FEATURE_SETS:
                    X = build_features(df_comp, fcols, df_met, fs)
    
                    # Multiclasse D0/D1/D2.
                    clf_multi = make_classifier()
                    clf_multi.fit(X[train_mask], y_train_multi)
                    pred_multi = clf_multi.predict(X[test_mask])
    
                    acc = accuracy_score(y_test_multi, pred_multi)
                    bacc = balanced_accuracy_score(y_test_multi, pred_multi)
                    mf1 = f1_score(y_test_multi, pred_multi, average="macro", labels=LABELS_MULTI, zero_division=0)
                    f1_por_dano = f1_score(y_test_multi, pred_multi, average=None, labels=LABELS_MULTI, zero_division=0)
    
                    resultados_multi.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "feature_set": fs,
                        "test_temp": float(T_test),
                        "accuracy": float(acc),
                        "balanced_accuracy": float(bacc),
                        "macro_f1": float(mf1),
                        "f1_dano0": float(f1_por_dano[0]),
                        "f1_dano1": float(f1_por_dano[1]),
                        "f1_dano2": float(f1_por_dano[2]),
                        "n_test": int(len(y_test_multi)),
                    })
    
                    for yt, yp in zip(y_test_multi, pred_multi):
                        pred_multi_rows.append({
                            "janela_id": janela_id,
                            "faixa_label": faixa_label,
                            "metodo": metodo,
                            "metodo_nome": NOME_METODO.get(metodo, metodo),
                            "feature_set": fs,
                            "test_temp": float(T_test),
                            "y_true": int(yt),
                            "y_pred": int(yp),
                        })
    
                    acumulados_multi[(metodo, fs)]["y_true"].extend(y_test_multi.tolist())
                    acumulados_multi[(metodo, fs)]["y_pred"].extend(pred_multi.tolist())
    
                    # Binário: sem dano vs com dano.
                    clf_bin = make_classifier()
                    clf_bin.fit(X[train_mask], y_train_bin)
                    pred_bin = clf_bin.predict(X[test_mask])
    
                    acc_bin = accuracy_score(y_test_bin, pred_bin)
                    bacc_bin = balanced_accuracy_score(y_test_bin, pred_bin)
                    mf1_bin = f1_score(y_test_bin, pred_bin, average="macro", labels=LABELS_BIN, zero_division=0)
                    recall_dano = recall_score(y_test_bin, pred_bin, pos_label=1, zero_division=0)
                    recall_sem = recall_score(y_test_bin, pred_bin, pos_label=0, zero_division=0)
                    precision_dano = precision_score(y_test_bin, pred_bin, pos_label=1, zero_division=0)
    
                    cm_bin = confusion_matrix(y_test_bin, pred_bin, labels=LABELS_BIN)
                    # cm_bin = [[TN, FP], [FN, TP]]
                    falso_saudavel = int(cm_bin[1, 0])
                    n_dano_real = int(cm_bin[1, :].sum())
                    taxa_falso_saudavel = falso_saudavel / max(n_dano_real, 1)
    
                    resultados_bin.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "feature_set": fs,
                        "test_temp": float(T_test),
                        "accuracy_bin": float(acc_bin),
                        "balanced_accuracy_bin": float(bacc_bin),
                        "macro_f1_bin": float(mf1_bin),
                        "recall_dano": float(recall_dano),
                        "recall_sem_dano": float(recall_sem),
                        "precision_dano": float(precision_dano),
                        "falso_saudavel_count": int(falso_saudavel),
                        "taxa_falso_saudavel": float(taxa_falso_saudavel),
                        "n_test": int(len(y_test_bin)),
                        "n_dano_real": int(n_dano_real),
                    })
    
                    for yt, yp in zip(y_test_bin, pred_bin):
                        pred_bin_rows.append({
                            "janela_id": janela_id,
                            "faixa_label": faixa_label,
                            "metodo": metodo,
                            "metodo_nome": NOME_METODO.get(metodo, metodo),
                            "feature_set": fs,
                            "test_temp": float(T_test),
                            "y_true_bin": int(yt),
                            "y_pred_bin": int(yp),
                        })
    
                    acumulados_bin[(metodo, fs)]["y_true"].extend(y_test_bin.tolist())
                    acumulados_bin[(metodo, fs)]["y_pred"].extend(pred_bin.tolist())
    
        return {
            "folds_multi": pd.DataFrame(resultados_multi),
            "folds_bin": pd.DataFrame(resultados_bin),
            "metricas_teste": pd.DataFrame(metricas_teste_rows),
            "pred_multi": pd.DataFrame(pred_multi_rows),
            "pred_bin": pd.DataFrame(pred_bin_rows),
            "acumulados_multi": acumulados_multi,
            "acumulados_bin": acumulados_bin,
        }
    
    
    # ============================================================
    # 9) RESUMOS
    # ============================================================
    
    def resumir_multiclasse(df_folds):
        if len(df_folds) == 0:
            return pd.DataFrame()
        return (
            df_folds
            .groupby(["janela_id", "faixa_min_khz", "faixa_max_khz", "faixa_label", "centro_khz", "tipo_extremo", "metodo", "metodo_nome", "feature_set"], as_index=False)
            .agg(
                accuracy_medio=("accuracy", "mean"),
                accuracy_std=("accuracy", "std"),
                balanced_accuracy_medio=("balanced_accuracy", "mean"),
                balanced_accuracy_std=("balanced_accuracy", "std"),
                macro_f1_medio=("macro_f1", "mean"),
                macro_f1_std=("macro_f1", "std"),
                f1_dano0_medio=("f1_dano0", "mean"),
                f1_dano1_medio=("f1_dano1", "mean"),
                f1_dano2_medio=("f1_dano2", "mean"),
                n_folds=("test_temp", "nunique"),
            )
            .sort_values("macro_f1_medio", ascending=False)
            .reset_index(drop=True)
        )
    
    
    def resumir_binario(df_folds_bin):
        if len(df_folds_bin) == 0:
            return pd.DataFrame()
        return (
            df_folds_bin
            .groupby(["janela_id", "faixa_min_khz", "faixa_max_khz", "faixa_label", "centro_khz", "tipo_extremo", "metodo", "metodo_nome", "feature_set"], as_index=False)
            .agg(
                accuracy_bin_medio=("accuracy_bin", "mean"),
                balanced_accuracy_bin_medio=("balanced_accuracy_bin", "mean"),
                macro_f1_bin_medio=("macro_f1_bin", "mean"),
                recall_dano_medio=("recall_dano", "mean"),
                recall_sem_dano_medio=("recall_sem_dano", "mean"),
                precision_dano_medio=("precision_dano", "mean"),
                falso_saudavel_total=("falso_saudavel_count", "sum"),
                taxa_falso_saudavel_media=("taxa_falso_saudavel", "mean"),
                n_folds=("test_temp", "nunique"),
            )
            .sort_values(["recall_dano_medio", "macro_f1_bin_medio"], ascending=False)
            .reset_index(drop=True)
        )
    
    
    # ============================================================
    # 10) GRÁFICOS
    # ============================================================
    
    def plot_ranking_multiclasse(df_resumo):
        sub = df_resumo[df_resumo["feature_set"] == FEATURE_SET_PREFERIDO].copy()
        if len(sub) == 0:
            return None
        top = sub.sort_values("macro_f1_medio", ascending=False).head(25).copy()
        top["label"] = top.apply(lambda r: f"J{int(r['janela_id'])} {r['faixa_label']} — {r['metodo_nome']}", axis=1)
        top = top.iloc[::-1]
    
        fig, ax = plt.subplots(figsize=(17, 11), dpi=300)
        y = np.arange(len(top))
        colors = [CORES_METODO.get(m, None) for m in top["metodo"]]
        ax.barh(y, top["macro_f1_medio"], xerr=top["macro_f1_std"].fillna(0), color=colors, edgecolor="black", linewidth=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(top["label"], fontsize=12)
        ax.set_xlabel("Macro-F1 médio — D0/D1/D2")
        ax.set_title(f"Ranking multiclasse — feature set: {FEATURE_SET_PREFERIDO}", pad=14)
        ax.set_xlim(0, min(1.05, max(0.2, top["macro_f1_medio"].max() + 0.15)))
        estilo_eixos(ax)
        fig.tight_layout()
        salvar_fig(fig, "ranking_multiclasse_macroF1_top25")
        plt.show()
        return fig
    
    
    def plot_ranking_binario(df_resumo_bin):
        sub = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].copy()
        if len(sub) == 0:
            return None
    
        top = sub.sort_values(["recall_dano_medio", "taxa_falso_saudavel_media", "macro_f1_bin_medio"], ascending=[False, True, False]).head(25).copy()
        top["label"] = top.apply(lambda r: f"J{int(r['janela_id'])} {r['faixa_label']} — {r['metodo_nome']}", axis=1)
        top = top.iloc[::-1]
    
        fig, ax = plt.subplots(figsize=(17, 11), dpi=300)
        y = np.arange(len(top))
        colors = [CORES_METODO.get(m, None) for m in top["metodo"]]
        ax.barh(y, top["recall_dano_medio"], color=colors, edgecolor="black", linewidth=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(top["label"], fontsize=12)
        ax.set_xlabel("Recall médio da classe com dano")
        ax.set_title("Ranking binário — capacidade de detectar dano", pad=14)
        ax.set_xlim(0, 1.05)
        estilo_eixos(ax)
        fig.tight_layout()
        salvar_fig(fig, "ranking_binario_recall_dano_top25")
        plt.show()
    
        fig, ax = plt.subplots(figsize=(17, 11), dpi=300)
        ax.barh(y, top["taxa_falso_saudavel_media"], color=colors, edgecolor="black", linewidth=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(top["label"], fontsize=12)
        ax.set_xlabel("Taxa média de falso saudável")
        ax.set_title("Erro crítico em SHM — dano real classificado como sem dano", pad=14)
        ax.set_xlim(0, max(0.05, min(1.05, top["taxa_falso_saudavel_media"].max() + 0.1)))
        estilo_eixos(ax)
        fig.tight_layout()
        salvar_fig(fig, "ranking_binario_falso_saudavel_top25")
        plt.show()
    
    
    def plot_scatter_binario(df_resumo_bin):
        sub = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].copy()
        if len(sub) == 0:
            return None
    
        fig, ax = plt.subplots(figsize=(12, 9), dpi=300)
        for metodo, gm in sub.groupby("metodo"):
            ax.scatter(
                gm["recall_sem_dano_medio"],
                gm["recall_dano_medio"],
                s=110,
                alpha=0.85,
                edgecolor="black",
                linewidth=0.7,
                color=CORES_METODO.get(metodo, None),
                label=NOME_METODO.get(metodo, metodo),
            )
            for _, r in gm.iterrows():
                ax.text(r["recall_sem_dano_medio"] + 0.008, r["recall_dano_medio"] + 0.008, f"J{int(r['janela_id'])}", fontsize=9)
    
        ax.set_xlabel("Recall sem dano / especificidade")
        ax.set_ylabel("Recall com dano / sensibilidade")
        ax.set_title("Trade-off binário: proteger contra falso saudável", pad=14)
        ax.set_xlim(-0.02, 1.05)
        ax.set_ylim(-0.02, 1.05)
        ax.legend(frameon=True, fontsize=10, loc="lower left")
        estilo_eixos(ax)
        fig.tight_layout()
        salvar_fig(fig, "scatter_binario_recall_sem_dano_vs_recall_dano")
        plt.show()
    
    
    def plot_confusion_from_predictions(df_pred, janela_id, metodo, feature_set, binario=False):
        sub = df_pred[(df_pred["janela_id"] == janela_id) & (df_pred["metodo"] == metodo) & (df_pred["feature_set"] == feature_set)].copy()
        if len(sub) == 0:
            print(f"⚠️ Sem predições para matriz: J{janela_id}, {metodo}, {feature_set}")
            return None
    
        if binario:
            y_true = sub["y_true_bin"].to_numpy(int)
            y_pred = sub["y_pred_bin"].to_numpy(int)
            labels = LABELS_BIN
            labels_txt = LABELS_BIN_TXT
            titulo_tipo = "binária"
        else:
            y_true = sub["y_true"].to_numpy(int)
            y_pred = sub["y_pred"].to_numpy(int)
            labels = LABELS_MULTI
            labels_txt = LABELS_MULTI_TXT
            titulo_tipo = "multiclasse"
    
        cm = confusion_matrix(y_true, y_pred, labels=labels)
    
        fig, ax = plt.subplots(figsize=(8.5, 7.2), dpi=300)
        im = ax.imshow(cm, aspect="auto")
        ax.set_xticks(np.arange(len(labels_txt)))
        ax.set_xticklabels(labels_txt)
        ax.set_yticks(np.arange(len(labels_txt)))
        ax.set_yticklabels(labels_txt)
        ax.set_xlabel("Predito")
        ax.set_ylabel("Real")
        ax.set_title(f"Matriz de confusão {titulo_tipo} — J{janela_id} — {NOME_METODO.get(metodo, metodo)}", pad=14)
    
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", fontsize=26, color="black")
    
        cbar = fig.colorbar(im, ax=ax)
        cbar.ax.tick_params(labelsize=14)
        fig.tight_layout()
        nome = f"matriz_confusao_{'binaria' if binario else 'multi'}_J{janela_id}_{safe_name(metodo)}_{safe_name(feature_set)}"
        salvar_fig(fig, nome, pasta=PASTA_CONFUSAO)
        plt.show()
        return fig
    
    
    def plot_metricas_por_temperatura(df_metricas, janela_id, metodos_escolhidos):
        sub = df_metricas[(df_metricas["janela_id"] == janela_id) & (df_metricas["metodo"].isin(metodos_escolhidos))].copy()
        if len(sub) == 0:
            return None
    
        for metrica in ["RMSD", "CCDM"]:
            for metodo in metodos_escolhidos:
                sm = sub[sub["metodo"] == metodo]
                if len(sm) == 0:
                    continue
    
                fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
                for d in DANOS:
                    g = (
                        sm[sm["falha"] == d]
                        .groupby("temperatura_c", as_index=False)[metrica]
                        .mean()
                        .sort_values("temperatura_c")
                    )
                    ax.plot(g["temperatura_c"], g[metrica], marker="o", linewidth=2.8, markersize=8, label=f"Dano {d}", color=CORES_DANO[d])
    
                faixa = sm["faixa_label"].iloc[0]
                ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.2, label=f"Ref. {REF_TEMP}°C")
                ax.set_xlabel("Temperatura (°C)")
                ax.set_ylabel(metrica)
                ax.set_title(f"{NOME_METODO.get(metodo, metodo)} — {metrica} por temperatura — J{janela_id} ({faixa})", pad=14)
                ax.legend(frameon=True, loc="best")
                estilo_eixos(ax)
                fig.tight_layout()
                salvar_fig(fig, f"{safe_name(metodo)}_J{janela_id}_{metrica}_por_temperatura", pasta=PASTA_METRICAS)
                plt.show()
    
    
    def escolher_temperatura_disponivel(df, T_alvo):
        temps_validas = []
        for T in sorted(df["temperatura_c"].dropna().unique()):
            ok = True
            for d in DANOS:
                if not np.any(np.isclose(df["temperatura_c"], T) & (df["falha"] == d)):
                    ok = False
                    break
            if ok:
                temps_validas.append(float(T))
        if len(temps_validas) == 0:
            raise ValueError("Nenhuma temperatura possui os três danos.")
        arr = np.asarray(temps_validas)
        return float(arr[np.argmin(np.abs(arr - T_alvo))])
    
    
    def plot_curvas_exemplo(df_base, row_window, metodos_rf_escolhidos, T_alvo=TEMP_EXEMPLO_CURVAS):
        fmin = float(row_window["faixa_min_khz"])
        fmax = float(row_window["faixa_max_khz"])
        janela_id = int(row_window["janela_id"])
        fcols, fHz = get_freq_columns(df_base, fmin, fmax)
        df_use = df_base[["temperatura_c", "falha"] + fcols].copy().reset_index(drop=True)
        y_ref, temp_ref_usada = curva_referencia_saudavel(df_use, fcols, REF_TEMP)
        fkhz = fHz / 1e3
    
        # Para visualização, treina os compensadores com todos os saudáveis.
        comps = {}
        for metodo_rf in metodos_rf_escolhidos:
            comps[metodo_rf] = compensar_rf_variante(df_use, fcols, y_ref, metodo_rf, mask_treino_sem_dano=(df_use["falha"].to_numpy(int) == 0))
        comps["Park"] = compensar_park(df_use, fcols, fHz, y_ref)
    
        T = escolher_temperatura_disponivel(df_use, T_alvo)
    
        fig, axes = plt.subplots(1, len(DANOS), figsize=(8 * len(DANOS), 7.5), dpi=300, sharey=True)
        if len(DANOS) == 1:
            axes = [axes]
    
        for ax, d in zip(axes, DANOS):
            mask = np.isclose(df_use["temperatura_c"], T) & (df_use["falha"] == d)
            idxs = np.where(mask.to_numpy())[0]
            if len(idxs) == 0:
                continue
            idx = int(idxs[0])
    
            y_orig = df_use.iloc[idx][fcols].to_numpy(float)
            ax.plot(fkhz, y_ref, "--", color="black", linewidth=2.0, label=f"Referência {format_temp(temp_ref_usada)}°C")
            ax.plot(fkhz, y_orig, color="lightcoral", linewidth=1.8, alpha=0.75, label=f"Original {format_temp(T)}°C")
    
            for metodo, dfc in comps.items():
                y_comp = dfc.iloc[idx][fcols].to_numpy(float)
                ax.plot(fkhz, y_comp, linewidth=2.2, color=CORES_METODO.get(metodo, None), label=NOME_METODO.get(metodo, metodo))
    
            ax.set_title(f"Dano {d}")
            ax.set_xlabel("Frequência (kHz)")
            estilo_eixos(ax)
    
        axes[0].set_ylabel("Impedância")
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=True, fontsize=12, bbox_to_anchor=(0.5, -0.07))
        fig.suptitle(f"Curvas exemplo — J{janela_id} ({format_faixa(fmin, fmax)}) — T = {format_temp(T)}°C", fontsize=28, y=1.02)
        fig.tight_layout(rect=[0, 0.08, 1, 0.95])
        salvar_fig(fig, f"curvas_exemplo_J{janela_id}_T{format_temp(T).replace('-', 'm')}", pasta=PASTA_CURVAS)
        plt.show()
    
    
    # ============================================================
    # 11) EXECUÇÃO PRINCIPAL
    # ============================================================
    
    def executar_teste():
        t0_total = time.time()
        print("=" * 100)
        print("TESTE — VARIAÇÕES DO RF LIVRE vs PARK EM JANELAS PEQUENAS PERTO DOS PICOS")
        print("=" * 100)
    
        print("\n🔹 Carregando base...")
        df_base = pd.read_pickle(ARQ_BASE).reset_index(drop=True)
    
        required = {"temperatura_c", "falha"}
        missing = required - set(df_base.columns)
        if missing:
            raise ValueError(f"A base está sem colunas obrigatórias: {missing}")
    
        df_base["temperatura_c"] = pd.to_numeric(df_base["temperatura_c"], errors="coerce")
        df_base["falha"] = pd.to_numeric(df_base["falha"], errors="coerce").astype(int)
    
        print(f"Total de amostras: {len(df_base)}")
        print(f"Temperaturas: {sorted(df_base['temperatura_c'].dropna().unique())}")
        print(f"Danos: {sorted(df_base['falha'].dropna().unique())}")
    
        print("\n🔹 Detectando janelas perto de picos/vales...")
        df_windows, fkhz_global, y_ref_global, temp_ref_usada = detectar_picos_vales(df_base)
        print(df_windows[["janela_id", "faixa_label", "centro_khz", "tipo_extremo", "n_freq_points"]].to_string(index=False))
        plot_janelas_picos(df_windows, fkhz_global, y_ref_global, temp_ref_usada)
    
        todos_multi = []
        todos_bin = []
        todas_metricas = []
        todas_pred_multi = []
        todas_pred_bin = []
        erros = []
    
        for _, roww in df_windows.iterrows():
            print("\n" + "-" * 100)
            print(f"🔹 Avaliando janela J{int(roww['janela_id'])}: {roww['faixa_label']} | centro {roww['centro_khz']:.2f} kHz | {roww['tipo_extremo']}")
            try:
                out = avaliar_uma_janela(df_base, roww)
                if len(out["folds_multi"]) > 0:
                    todos_multi.append(out["folds_multi"])
                if len(out["folds_bin"]) > 0:
                    todos_bin.append(out["folds_bin"])
                if len(out["metricas_teste"]) > 0:
                    todas_metricas.append(out["metricas_teste"])
                if len(out["pred_multi"]) > 0:
                    todas_pred_multi.append(out["pred_multi"])
                if len(out["pred_bin"]) > 0:
                    todas_pred_bin.append(out["pred_bin"])
            except Exception as e:
                print(f"⚠️ Erro na janela J{int(roww['janela_id'])}: {e}")
                erros.append({
                    "janela_id": int(roww["janela_id"]),
                    "faixa_label": roww["faixa_label"],
                    "erro": str(e),
                })
    
        if len(todos_multi) == 0:
            raise RuntimeError("Nenhum resultado multiclasse foi gerado.")
    
        df_multi = pd.concat(todos_multi, ignore_index=True)
        df_bin = pd.concat(todos_bin, ignore_index=True) if len(todos_bin) else pd.DataFrame()
        df_metricas = pd.concat(todas_metricas, ignore_index=True) if len(todas_metricas) else pd.DataFrame()
        df_pred_multi = pd.concat(todas_pred_multi, ignore_index=True) if len(todas_pred_multi) else pd.DataFrame()
        df_pred_bin = pd.concat(todas_pred_bin, ignore_index=True) if len(todas_pred_bin) else pd.DataFrame()
    
        df_resumo_multi = resumir_multiclasse(df_multi)
        df_resumo_bin = resumir_binario(df_bin)
    
        # Salvar CSVs.
        arquivos = {
            "resultados_folds_multiclasse.csv": df_multi,
            "resultados_folds_binario.csv": df_bin,
            "resumo_multiclasse_por_janela.csv": df_resumo_multi,
            "resumo_binario_por_janela.csv": df_resumo_bin,
            "metricas_teste_por_amostra.csv": df_metricas,
            "predicoes_multiclasse.csv": df_pred_multi,
            "predicoes_binario.csv": df_pred_bin,
        }
        for nome, df_ in arquivos.items():
            if df_ is not None and len(df_) > 0:
                path = os.path.join(PASTA_CSV, nome)
                df_.to_csv(path, index=False)
                print(f"✅ CSV salvo: {path}")
    
        if len(erros) > 0:
            pd.DataFrame(erros).to_csv(os.path.join(PASTA_CSV, "erros_janelas.csv"), index=False)
    
        # Gráficos principais.
        plot_ranking_multiclasse(df_resumo_multi)
        if len(df_resumo_bin) > 0:
            plot_ranking_binario(df_resumo_bin)
            plot_scatter_binario(df_resumo_bin)
    
        # Melhor método/janela multiclasse e binário.
        pref_multi = df_resumo_multi[df_resumo_multi["feature_set"] == FEATURE_SET_PREFERIDO].sort_values("macro_f1_medio", ascending=False)
        print("\n🏆 Top 20 multiclasse — feature_set preferido:")
        print(pref_multi[[
            "janela_id", "faixa_label", "metodo_nome", "macro_f1_medio", "balanced_accuracy_medio",
            "f1_dano0_medio", "f1_dano1_medio", "f1_dano2_medio"
        ]].head(20).to_string(index=False))
    
        best_multi = pref_multi.iloc[0]
        best_multi_j = int(best_multi["janela_id"])
        best_multi_metodo = str(best_multi["metodo"])
    
        print("\n🏆 Melhor geral multiclasse:")
        print(best_multi.to_string())
    
        if len(df_resumo_bin) > 0:
            pref_bin = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].sort_values(
                ["recall_dano_medio", "taxa_falso_saudavel_media", "macro_f1_bin_medio"],
                ascending=[False, True, False]
            )
            print("\n🏆 Top 20 binário — sem dano vs com dano:")
            print(pref_bin[[
                "janela_id", "faixa_label", "metodo_nome", "recall_dano_medio", "recall_sem_dano_medio",
                "macro_f1_bin_medio", "falso_saudavel_total", "taxa_falso_saudavel_media"
            ]].head(20).to_string(index=False))
    
            best_bin = pref_bin.iloc[0]
            best_bin_j = int(best_bin["janela_id"])
            best_bin_metodo = str(best_bin["metodo"])
            print("\n🏆 Melhor geral binário:")
            print(best_bin.to_string())
        else:
            best_bin_j = best_multi_j
            best_bin_metodo = best_multi_metodo
    
        # Matrizes de confusão: melhor RF, melhor geral e Park na mesma janela.
        melhores_para_plot = []
    
        # Melhor RF multiclasse.
        pref_rf = pref_multi[pref_multi["metodo"].isin(METODOS_RF)]
        if len(pref_rf) > 0:
            best_rf = pref_rf.iloc[0]
            melhores_para_plot.append((int(best_rf["janela_id"]), str(best_rf["metodo"])))
    
        melhores_para_plot.append((best_multi_j, best_multi_metodo))
        melhores_para_plot.append((best_multi_j, "Park"))
    
        # Remove duplicatas mantendo ordem.
        vistos = set()
        unicos = []
        for item in melhores_para_plot:
            if item not in vistos:
                vistos.add(item)
                unicos.append(item)
    
        for j, metodo in unicos:
            plot_confusion_from_predictions(df_pred_multi, j, metodo, FEATURE_SET_PREFERIDO, binario=False)
            if len(df_pred_bin) > 0:
                plot_confusion_from_predictions(df_pred_bin, j, metodo, FEATURE_SET_PREFERIDO, binario=True)
    
        # Métricas por temperatura e curvas exemplo para os melhores.
        metodos_exemplo = []
        if len(pref_rf) > 0:
            metodos_exemplo.append(str(pref_rf.iloc[0]["metodo"]))
        if best_multi_metodo not in metodos_exemplo:
            metodos_exemplo.append(best_multi_metodo)
        if best_bin_metodo not in metodos_exemplo:
            metodos_exemplo.append(best_bin_metodo)
        if "Park" not in metodos_exemplo:
            metodos_exemplo.append("Park")
    
        metodos_exemplo = [m for m in metodos_exemplo if m in METODOS_TODOS]
    
        # Usa a janela do melhor RF se houver, senão a melhor geral.
        janela_plot = int(pref_rf.iloc[0]["janela_id"]) if len(pref_rf) > 0 else best_multi_j
        row_window_plot = df_windows[df_windows["janela_id"] == janela_plot].iloc[0]
    
        if len(df_metricas) > 0:
            plot_metricas_por_temperatura(df_metricas, janela_plot, metodos_exemplo)
    
        metodos_rf_exemplo = [m for m in metodos_exemplo if m in METODOS_RF]
        if len(metodos_rf_exemplo) == 0 and len(pref_rf) > 0:
            metodos_rf_exemplo = [str(pref_rf.iloc[0]["metodo"])]
        plot_curvas_exemplo(df_base, row_window_plot, metodos_rf_exemplo, TEMP_EXEMPLO_CURVAS)
    
        print("\n" + "=" * 100)
        print("✅ TESTE FINALIZADO")
        print(f"📁 Pasta de saída: {PASTA_SAIDA}")
        print(f"⏱️ Tempo total: {time.time() - t0_total:.1f} s")
        print("=" * 100)
    
        return {
            "df_windows": df_windows,
            "df_multi": df_multi,
            "df_bin": df_bin,
            "df_metricas": df_metricas,
            "df_pred_multi": df_pred_multi,
            "df_pred_bin": df_pred_bin,
            "df_resumo_multi": df_resumo_multi,
            "df_resumo_bin": df_resumo_bin,
        }
    
    
    # ============================================================
    # 12) RODAR
    # ============================================================
    
    if __name__ == "__main__":
        resultados = executar_teste()
